In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("surat_house_price.csv")


In [4]:
df["square_feet"] = df["square_feet"].str.replace("sqft", "", case=False)
df["square_feet"] = df["square_feet"].str.replace(",", "").str.strip()
df["square_feet"] = pd.to_numeric(df["square_feet"], errors="coerce")


In [5]:
df["price_per_sqft"] = df["price_per_sqft"].str.replace("₹", "")
df["price_per_sqft"] = df["price_per_sqft"].str.replace("per sqft", "", case=False)
df["price_per_sqft"] = df["price_per_sqft"].str.replace(",", "").str.strip()
df["price_per_sqft"] = pd.to_numeric(df["price_per_sqft"], errors="coerce")


In [6]:
def clean_price(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace("₹", "").replace(",", "").strip()
    if "Cr" in val:
        return float(val.replace("Cr", "").strip()) * 100  # Convert Cr to Lacs
    elif "Lac" in val:
        return float(val.replace("Lac", "").strip())
    return np.nan

df["price_in_lacs"] = df["price"].apply(clean_price)


In [7]:
# Extract BHK number (e.g., '2 BHK' -> 2.0)
df["bhk"] = df["property_name"].str.extract(r'(\d+)\s*BHK').astype(float)

# Extract Area Location (e.g., 'in Dindoli Surat' -> 'Dindoli')
df["location"] = df["property_name"].str.extract(r'for\s+Sale\s+in\s+(.*?)\s+Surat')[0].str.strip()


In [8]:
# Extract floor number (e.g., '5 out of 10' -> current_floor=5.0, total_floors=10.0)
df["current_floor"] = df["floor"].str.extract(r'(\d+)\s+out\s+of').astype(float)
df["total_floors"] = df["floor"].str.extract(r'out\s+of\s+(\d+)').astype(float)


In [9]:
# Fill missing price_per_sqft using: (price_in_lacs * 100,000) / square_feet
missing_mask = df["price_per_sqft"].isna() & df["square_feet"].notna() & df["price_in_lacs"].notna()
df.loc[missing_mask, "price_per_sqft"] = (df.loc[missing_mask, "price_in_lacs"] * 100000) / df.loc[missing_mask, "square_feet"]

# Fill missing furnishing with Mode
df["furnishing"] = df["furnishing"].fillna(df["furnishing"].mode()[0])


In [10]:
# Fill missing price_per_sqft using: (price_in_lacs * 100,000) / square_feet
missing_mask = df["price_per_sqft"].isna() & df["square_feet"].notna() & df["price_in_lacs"].notna()
df.loc[missing_mask, "price_per_sqft"] = (df.loc[missing_mask, "price_in_lacs"] * 100000) / df.loc[missing_mask, "square_feet"]

# Fill missing furnishing with Mode
df["furnishing"] = df["furnishing"].fillna(df["furnishing"].mode()[0])


In [11]:
print(df[["location", "bhk", "square_feet", "price_in_lacs", "price_per_sqft", "furnishing"]].head(10))


                     location  bhk  square_feet  price_in_lacs  \
0                     Dindoli  2.0        644.0           33.8   
1                      Althan  2.0       1278.0           45.4   
2                     Pal Gam  2.0       1173.0           44.6   
3                Jahangirabad  2.0        700.0           47.0   
4   Orchid Fantasia, Palanpur  2.0       1250.0           45.0   
5  Anand Aspire, Jahangirabad  2.0       1265.0           43.2   
6                     Dindoli  3.0       1404.0           42.1   
7                        Vesu  NaN        700.0           44.1   
8   Orchid Gardenia, Palanpur  2.0       1180.0           44.3   
9                    Palanpur  2.0        720.0           40.0   

   price_per_sqft      furnishing  
0     2891.000000     Unfurnished  
1     3551.000000     Unfurnished  
2     3800.000000  Semi-Furnished  
3     3966.000000     Unfurnished  
4     3600.000000               2  
5     3411.000000    Anand Aspire  
6     2998.575499    